# Company Fundamentals

This notebook pulls the income statement, cash flow statement, and balance sheet for a ticker using Alpha Vantage.

The current ticker below is `IBM`. Change the `ticker_symbol` value in the data cell if you want a different company.

The notebook reads `ALPHAVANTAGE_API_KEY` from the workspace `.env` file through the project config helper, requests annual reports for each statement, and builds a small business-health dashboard from the last 10 years of fundamentals.

In [5]:
%pip install -q pandas requests plotly


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import sys
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

ROOT = Path.cwd().resolve()
WORKSPACE_ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
SRC_PATH = WORKSPACE_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from finance.config import build_settings, get_required_alphavantage_api_key

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 140)

ticker_symbol = "IBM"
requested_years = 10
settings = build_settings()
api_key = get_required_alphavantage_api_key(settings)

statement_functions = {
    "Income Statement": "INCOME_STATEMENT",
    "Cash Flow": "CASH_FLOW",
    "Balance Sheet": "BALANCE_SHEET",
}

report_keys = {
    "Income Statement": "annualReports",
    "Cash Flow": "annualReports",
    "Balance Sheet": "annualReports",
}


def call_alpha_vantage(params: dict, attempts: int = 3) -> dict:
    for attempt in range(attempts):
        response = requests.get(
            "https://www.alphavantage.co/query",
            params={**params, "apikey": api_key},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()

        if not isinstance(payload, dict):
            raise ValueError(f"Unexpected Alpha Vantage payload type: {type(payload)!r}")

        if "Note" in payload or "Information" in payload:
            if attempt == attempts - 1:
                message = payload.get("Note") or payload.get("Information")
                raise RuntimeError(message)
            time.sleep(1.5 * (attempt + 1))
            continue

        if "Error Message" in payload:
            raise ValueError(payload["Error Message"])

        return payload

    raise RuntimeError(f"Failed to fetch payload for {params}")


def fetch_statement(function_name: str, symbol: str) -> dict:
    return call_alpha_vantage({"function": function_name, "symbol": symbol})


def search_symbols(keyword: str) -> pd.DataFrame:
    payload = call_alpha_vantage({"function": "SYMBOL_SEARCH", "keywords": keyword})
    matches = pd.DataFrame(payload.get("bestMatches", []))
    if matches.empty:
        return matches

    renamed = matches.rename(
        columns={
            "1. symbol": "symbol",
            "2. name": "name",
            "3. type": "type",
            "4. region": "region",
            "8. currency": "currency",
            "9. matchScore": "match_score",
        }
    )
    columns = [column for column in ["symbol", "name", "type", "region", "currency", "match_score"] if column in renamed.columns]
    return renamed[columns]


def clean_statement(records: list[dict], date_key: str) -> pd.DataFrame:
    if not records:
        return pd.DataFrame()

    frame = pd.DataFrame(records).copy()
    frame[date_key] = pd.to_datetime(frame[date_key], errors="coerce")
    frame = frame.dropna(subset=[date_key]).sort_values(date_key, ascending=False)
    frame = frame.set_index(date_key)

    for column in frame.columns:
        converted = pd.to_numeric(frame[column], errors="coerce")
        if converted.notna().any():
            frame[column] = converted

    return frame


statement_payloads = {}
for name, function_name in statement_functions.items():
    statement_payloads[name] = fetch_statement(function_name, ticker_symbol)
    time.sleep(1.1)

annual_statements = {
    name: clean_statement(payload.get(report_keys[name], []), "fiscalDateEnding")
    for name, payload in statement_payloads.items()
}

annual_available = {name: len(frame.index) for name, frame in annual_statements.items()}
symbol_suggestions = pd.DataFrame()

print("Ticker:", ticker_symbol)
print("Requested annual history:", requested_years, "years")
print("Annual periods returned by Alpha Vantage:", annual_available)

if not any(count > 0 for count in annual_available.values()):
    print(
        "No annual statements were returned for this symbol from Alpha Vantage. "
        "This usually means the symbol is not the exact listing Alpha Vantage expects."
    )
    time.sleep(1.1)
    symbol_suggestions = search_symbols(ticker_symbol)
    if symbol_suggestions.empty:
        print("No close symbol matches were returned either.")
    else:
        print("Closest Alpha Vantage symbol matches:")
        display(symbol_suggestions.head(10))
else:
    for name, frame in annual_statements.items():
        print(f"\n{name} - annual history")
        if frame.empty:
            print("No annual data returned")
            continue

        annual_view = frame.head(requested_years).copy()
        annual_view.index = annual_view.index.strftime("%Y-%m-%d")
        display(annual_view)

        if len(frame.index) < requested_years:
            print(
                f"Alpha Vantage returned {len(frame.index)} annual periods for {name.lower()}, "
                f"which is fewer than the requested {requested_years} years."
            )

statements = annual_statements

Ticker: IBM
Requested annual history: 10 years
Annual periods returned by Alpha Vantage: {'Income Statement': 20, 'Cash Flow': 20, 'Balance Sheet': 20}

Income Statement - annual history


,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,...,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
fiscalDateEnding,,,,,,,,,,,,,
2025-12-31,USD,40185000000,67535000000,27350000000,27350000000,10325000000,...,None,1.057100e+10,None,12263000000,17284000000,10593000000
2024-12-31,USD,35551000000,62753000000,27201000000,27201000000,10074000000,...,None,6.015000e+09,None,7509000000,12176000000,6023000000
2023-12-31,USD,34300000000,61860000000,27560000000,27560000000,7514000000,...,None,7.099000e+09,None,7514000000,7514000000,7502000000
2022-12-31,USD,32687000000,60530000000,27842000000,27842000000,8174000000,...,None,1.782000e+09,None,2372000000,7174000000,1640000000
2021-12-31,USD,31486000000,57351000000,25865000000,25865000000,6832000000,...,None,4.712000e+09,None,7021000000,13438000000,5742000000
2020-12-31,USD,35575000000,73621000000,38046000000,38046000000,6895000000,...,None,5.501000e+09,None,6014000000,12709000000,5590000000
2019-12-31,USD,31533000000,57714000000,26181000000,26181000000,7538000000,...,None,9.435000e+09,None,8550000000,14609000000,9431000000
2018-12-31,USD,36936000000,79591000000,42655000000,42655000000,13217000000,...,None,8.723000e+09,None,12065000000,16545000000,8728000000
2017-12-31,USD,36943000000,79139000000,42196000000,42196000000,13139000000,...,None,5.758000e+09,None,12015000000,16556000000,5753000000



Cash Flow - annual history


,reportedCurrency,operatingCashflow,paymentsForOperatingActivities,proceedsFromOperatingActivities,changeInOperatingLiabilities,changeInOperatingAssets,...,proceedsFromRepurchaseOfEquity,proceedsFromSaleOfTreasuryStock,stockBasedCompensation,changeInCashAndCashEquivalents,changeInExchangeRate,netIncome
fiscalDateEnding,,,,,,,,,,,,,
2025-12-31,USD,13192000000,None,None,None,None,...,-1018000000,None,1715000000,NaN,NaN,10571000000
2024-12-31,USD,13445000000,None,None,None,None,...,-651000000,None,1311000000,NaN,NaN,6023000000
2023-12-31,USD,13432000000,None,None,None,None,...,-416000000,None,1091000000,NaN,NaN,6925000000
2022-12-31,USD,10435000000,None,None,None,None,...,-407000000,None,987000000,NaN,NaN,1639000000
2021-12-31,USD,12796000000,None,None,None,None,...,-319000000,None,982000000,-6.533000e+09,NaN,5743000000
2020-12-31,USD,18197000000,None,None,None,None,...,-210000000,None,937000000,5.448000e+09,NaN,5590000000
2019-12-31,USD,14770000000,None,None,None,None,...,-1633000000,None,679000000,-3.124000e+09,NaN,9431000000
2018-12-31,USD,15247000000,None,None,None,None,...,-4614000000,None,510000000,-1.350000e+08,NaN,8728000000
2017-12-31,USD,16724000000,None,None,None,None,...,-4533000000,None,534000000,4.146000e+09,937000000.0,5753000000



Balance Sheet - annual history


,reportedCurrency,totalAssets,totalCurrentAssets,cashAndCashEquivalentsAtCarryingValue,cashAndShortTermInvestments,inventory,...,otherNonCurrentLiabilities,totalShareholderEquity,treasuryStock,retainedEarnings,commonStock,commonStockSharesOutstanding
fiscalDateEnding,,,,,,,,,,,,,
2025-12-31,USD,151880000000,35860000000,13641000000,13641000000,1220000000,...,1.068000e+09,32648000000,NaN,155648000000,63318000000,948700000
2024-12-31,USD,137175000000,34482000000,13947000000,13947000000,1289000000,...,9.810000e+08,27307000000,NaN,151163000000,61380000000,937200000
2023-12-31,USD,135241000000,32908000000,13068000000,13068000000,1161000000,...,1.164000e+09,22533000000,NaN,151276000000,59643000000,922073828
2022-12-31,USD,127243000000,29118000000,7886000000,7886000000,1552000000,...,1.224300e+10,21944000000,NaN,149825000000,58343000000,912269062
2021-12-31,USD,132001000000,29539000000,6650000000,6650000000,1649000000,...,1.399500e+10,18996000000,-1.693920e+11,154209000000,57319000000,904641001
2020-12-31,USD,155970000000,39165000000,13212000000,13212000000,1839000000,...,3.671900e+10,20725000000,-1.693390e+11,162717000000,56556000000,896600000
2019-12-31,USD,152186000000,38420000000,8172000000,8172000000,1619000000,...,3.554700e+10,20985000000,-1.694130e+11,162954000000,55895000000,892800000
2018-12-31,USD,123382000000,49146000000,11379000000,11379000000,1682000000,...,3.262100e+10,16930000000,-1.680710e+11,159206000000,55151000000,916300000
2017-12-31,USD,125355000000,49735000000,11972000000,11972000000,1583000000,...,2.668500e+10,17724000000,-1.635070e+11,153126000000,54566000000,937400000


In [11]:
import pandas as pd
import plotly.express as px

income_statement = statements.get("Income Statement", pd.DataFrame())

if income_statement.empty:
    raise ValueError("Income statement data is not available. Run Cell 3 first.")

if "ebit" not in income_statement.columns:
    similar_columns = [column for column in income_statement.columns if "ebit" in column.lower()]
    raise KeyError(f"EBIT column was not found. Similar columns: {similar_columns}")

if "totalRevenue" not in income_statement.columns:
    raise KeyError("totalRevenue column was not found, so EBIT margin cannot be calculated.")

ebit_history = (
    income_statement[["ebit", "totalRevenue"]]
    .dropna(subset=["ebit"])
    .sort_index()
    .tail(requested_years)
    .copy()
)

if ebit_history.empty:
    raise ValueError("No EBIT history is available for the requested period.")

ebit_history["ebit_billions"] = ebit_history["ebit"] / 1_000_000_000
ebit_history["ebit_yoy_pct"] = ebit_history["ebit"].pct_change() * 100
ebit_history["ebit_margin_pct"] = (ebit_history["ebit"] / ebit_history["totalRevenue"]) * 100

start_ebit = ebit_history["ebit"].iloc[0]
end_ebit = ebit_history["ebit"].iloc[-1]
years = max(len(ebit_history.index) - 1, 1)
ebit_cagr_pct = (((end_ebit / start_ebit) ** (1 / years)) - 1) * 100 if start_ebit > 0 else float("nan")
latest = ebit_history.iloc[-1]
previous = ebit_history.iloc[-2] if len(ebit_history) > 1 else None

summary_lines = [
    f"Ticker: {ticker_symbol}",
    f"EBIT history available: {len(ebit_history)} annual periods",
    f"Latest EBIT: ${latest['ebit_billions']:.2f}B",
    f"Latest EBIT margin: {latest['ebit_margin_pct']:.1f}%",
]

if previous is not None and pd.notna(latest["ebit_yoy_pct"]):
    summary_lines.append(f"Latest YoY EBIT change: {latest['ebit_yoy_pct']:.1f}%")

if pd.notna(ebit_cagr_pct):
    summary_lines.append(f"EBIT CAGR over sample: {ebit_cagr_pct:.1f}%")

print("\n".join(summary_lines))
display(ebit_history[["ebit_billions", "ebit_yoy_pct", "ebit_margin_pct"]].round(2))

plot_frame = ebit_history.reset_index().rename(columns={"fiscalDateEnding": "date"})
plot_frame["date"] = pd.to_datetime(plot_frame["date"])

fig = px.line(
    plot_frame,
    x="date",
    y="ebit_billions",
    markers=True,
    title=f"{ticker_symbol} EBIT Over Time",
    labels={"date": "Fiscal year", "ebit_billions": "EBIT ($ billions)"},
)
fig.update_layout(height=500)
fig.show()

Ticker: IBM
EBIT history available: 10 annual periods
Latest EBIT: $12.26B
Latest EBIT margin: 18.2%
Latest YoY EBIT change: 63.3%
EBIT CAGR over sample: -0.6%


,ebit_billions,ebit_yoy_pct,ebit_margin_pct
fiscalDateEnding,,,
2016-12-31,12.96,NaN,16.22
2017-12-31,12.02,-7.29,15.18
2018-12-31,12.06,0.42,15.16
2019-12-31,8.55,-29.13,14.81
2020-12-31,6.01,-29.66,8.17
2021-12-31,7.02,16.74,12.24
2022-12-31,2.37,-66.22,3.92
2023-12-31,7.51,216.78,12.15
2024-12-31,7.51,-0.07,11.97


In [12]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

income_statement = statements.get("Income Statement", pd.DataFrame())
cash_flow = statements.get("Cash Flow", pd.DataFrame())
balance_sheet = statements.get("Balance Sheet", pd.DataFrame())

if income_statement.empty or cash_flow.empty or balance_sheet.empty:
    raise ValueError("Income statement, cash flow, and balance sheet data are required. Run Cell 3 first.")


def get_first_series(frame: pd.DataFrame, candidates: list[str], label: str, required: bool = True) -> pd.Series:
    for column in candidates:
        if column in frame.columns:
            series = pd.to_numeric(frame[column], errors="coerce")
            series.name = column
            return series
    if required:
        raise KeyError(f"Could not find {label}. Tried columns: {candidates}")
    return pd.Series(index=frame.index, dtype="float64", name=label)


def cagr_percent(series: pd.Series) -> float:
    clean = series.dropna()
    if len(clean) < 2 or clean.iloc[0] <= 0 or clean.iloc[-1] <= 0:
        return float("nan")
    periods = len(clean) - 1
    return ((clean.iloc[-1] / clean.iloc[0]) ** (1 / periods) - 1) * 100


def classify_health(value: float, green_floor: float, amber_floor: float, higher_is_better: bool = True) -> str:
    if pd.isna(value):
        return "unknown"
    if higher_is_better:
        if value >= green_floor:
            return "green"
        if value >= amber_floor:
            return "amber"
        return "red"
    if value <= green_floor:
        return "green"
    if value <= amber_floor:
        return "amber"
    return "red"


revenue = get_first_series(income_statement, ["totalRevenue"], "revenue")
ebit = get_first_series(income_statement, ["ebit"], "ebit")
net_income = get_first_series(income_statement, ["netIncome"], "net income")
operating_cash_flow = get_first_series(cash_flow, ["operatingCashflow"], "operating cash flow")
capital_expenditures_raw = get_first_series(cash_flow, ["capitalExpenditures"], "capital expenditures", required=False)
cash_balance = get_first_series(
    balance_sheet,
    ["cashAndCashEquivalentsAtCarryingValue", "cashAndShortTermInvestments"],
    "cash balance",
    required=False,
 )
total_debt_direct = get_first_series(balance_sheet, ["shortLongTermDebtTotal", "totalDebt"], "total debt", required=False)
short_term_debt = get_first_series(balance_sheet, ["shortTermDebt", "currentDebt"], "short-term debt", required=False)
long_term_debt = get_first_series(balance_sheet, ["longTermDebt", "longTermDebtNoncurrent"], "long-term debt", required=False)
shareholder_equity = get_first_series(balance_sheet, ["totalShareholderEquity"], "shareholder equity", required=False)

capital_expenditures = capital_expenditures_raw.abs()
total_debt = total_debt_direct.where(total_debt_direct.notna(), short_term_debt.fillna(0) + long_term_debt.fillna(0))

fundamentals = pd.concat(
    [
        revenue.rename("revenue"),
        ebit.rename("ebit"),
        net_income.rename("net_income"),
        operating_cash_flow.rename("operating_cash_flow"),
        capital_expenditures.rename("capital_expenditures"),
        cash_balance.rename("cash_balance"),
        total_debt.rename("total_debt"),
        shareholder_equity.rename("shareholder_equity"),
    ],
    axis=1,
).sort_index().tail(requested_years)

fundamentals["free_cash_flow"] = fundamentals["operating_cash_flow"] - fundamentals["capital_expenditures"]
fundamentals["net_debt"] = fundamentals["total_debt"] - fundamentals["cash_balance"]
fundamentals["revenue_billions"] = fundamentals["revenue"] / 1_000_000_000
fundamentals["ebit_billions"] = fundamentals["ebit"] / 1_000_000_000
fundamentals["net_income_billions"] = fundamentals["net_income"] / 1_000_000_000
fundamentals["free_cash_flow_billions"] = fundamentals["free_cash_flow"] / 1_000_000_000
fundamentals["cash_billions"] = fundamentals["cash_balance"] / 1_000_000_000
fundamentals["debt_billions"] = fundamentals["total_debt"] / 1_000_000_000
fundamentals["net_debt_billions"] = fundamentals["net_debt"] / 1_000_000_000
fundamentals["ebit_margin_pct"] = fundamentals["ebit"] / fundamentals["revenue"] * 100
fundamentals["net_margin_pct"] = fundamentals["net_income"] / fundamentals["revenue"] * 100
fundamentals["fcf_margin_pct"] = fundamentals["free_cash_flow"] / fundamentals["revenue"] * 100
fundamentals["revenue_yoy_pct"] = fundamentals["revenue"].pct_change() * 100
fundamentals["net_income_yoy_pct"] = fundamentals["net_income"].pct_change() * 100
fundamentals["free_cash_flow_yoy_pct"] = fundamentals["free_cash_flow"].pct_change() * 100
fundamentals["debt_to_equity"] = fundamentals["total_debt"] / fundamentals["shareholder_equity"]

latest_fundamentals = fundamentals.dropna(how="all").iloc[-1]
latest_year = fundamentals.dropna(how="all").index[-1].year

growth_score = classify_health(cagr_percent(fundamentals["revenue"]), 3.0, 0.0)
margin_score = classify_health(latest_fundamentals["ebit_margin_pct"], 15.0, 8.0)
cash_conversion_ratio = latest_fundamentals["operating_cash_flow"] / latest_fundamentals["net_income"] if latest_fundamentals["net_income"] not in (0, np.nan) else float("nan")
cash_conversion_score = classify_health(cash_conversion_ratio, 1.0, 0.75)
fcf_positive_years = int((fundamentals["free_cash_flow"] > 0).sum())
fcf_score = classify_health(fcf_positive_years / len(fundamentals.index), 0.8, 0.6)
debt_to_ebit = latest_fundamentals["net_debt"] / latest_fundamentals["ebit"] if latest_fundamentals["ebit"] not in (0, np.nan) else float("nan")
leverage_score = classify_health(debt_to_ebit, 1.5, 3.0, higher_is_better=False)

scorecard = pd.DataFrame(
    [
        {"metric": "Revenue CAGR", "value": cagr_percent(fundamentals["revenue"]), "unit": "%", "status": growth_score},
        {"metric": "Latest EBIT margin", "value": latest_fundamentals["ebit_margin_pct"], "unit": "%", "status": margin_score},
        {"metric": "Cash conversion", "value": cash_conversion_ratio, "unit": "x", "status": cash_conversion_score},
        {"metric": "FCF positive years", "value": fcf_positive_years, "unit": f"/{len(fundamentals.index)}", "status": fcf_score},
        {"metric": "Net debt to EBIT", "value": debt_to_ebit, "unit": "x", "status": leverage_score},
    ]
 )

summary_text = [
    f"Ticker: {ticker_symbol}",
    f"Health window: {fundamentals.index[0].year} to {latest_year}",
    f"Revenue CAGR: {cagr_percent(fundamentals['revenue']):.1f}%",
    f"Latest EBIT margin: {latest_fundamentals['ebit_margin_pct']:.1f}%",
    f"Latest net margin: {latest_fundamentals['net_margin_pct']:.1f}%",
    f"Latest free cash flow margin: {latest_fundamentals['fcf_margin_pct']:.1f}%",
    f"Cash conversion: {cash_conversion_ratio:.2f}x",
    f"Net debt to EBIT: {debt_to_ebit:.2f}x",
    f"Positive free cash flow years: {fcf_positive_years} of {len(fundamentals.index)}",
]

print("\n".join(summary_text))

display_columns = [
    "revenue_billions",
    "ebit_billions",
    "net_income_billions",
    "free_cash_flow_billions",
    "revenue_yoy_pct",
    "net_income_yoy_pct",
    "free_cash_flow_yoy_pct",
    "ebit_margin_pct",
    "net_margin_pct",
    "fcf_margin_pct",
    "cash_billions",
    "debt_billions",
    "net_debt_billions",
]
display(fundamentals[display_columns].round(2))
display(scorecard.assign(value=scorecard["value"].round(2)))

plot_frame = fundamentals.reset_index().rename(columns={"fiscalDateEnding": "date"})

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Scale And Cash Generation",
        "Margins",
        "Balance Sheet",
        "Growth Rates",
    ),
 )

for column, name in [
    ("revenue_billions", "Revenue"),
    ("ebit_billions", "EBIT"),
    ("net_income_billions", "Net Income"),
    ("free_cash_flow_billions", "Free Cash Flow"),
]:
    fig.add_trace(
        go.Scatter(x=plot_frame["date"], y=plot_frame[column], mode="lines+markers", name=name),
        row=1,
        col=1,
    )

for column, name in [
    ("ebit_margin_pct", "EBIT Margin"),
    ("net_margin_pct", "Net Margin"),
    ("fcf_margin_pct", "FCF Margin"),
]:
    fig.add_trace(
        go.Scatter(x=plot_frame["date"], y=plot_frame[column], mode="lines+markers", name=name),
        row=1,
        col=2,
    )

for column, name in [
    ("cash_billions", "Cash"),
    ("debt_billions", "Debt"),
    ("net_debt_billions", "Net Debt"),
]:
    fig.add_trace(
        go.Bar(x=plot_frame["date"], y=plot_frame[column], name=name),
        row=2,
        col=1,
    )

for column, name in [
    ("revenue_yoy_pct", "Revenue YoY"),
    ("net_income_yoy_pct", "Net Income YoY"),
    ("free_cash_flow_yoy_pct", "FCF YoY"),
]:
    fig.add_trace(
        go.Scatter(x=plot_frame["date"], y=plot_frame[column], mode="lines+markers", name=name),
        row=2,
        col=2,
    )

fig.update_layout(
    title=f"{ticker_symbol} Business Health Dashboard",
    height=900,
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_yaxes(title_text="$ billions", row=1, col=1)
fig.update_yaxes(title_text="%", row=1, col=2)
fig.update_yaxes(title_text="$ billions", row=2, col=1)
fig.update_yaxes(title_text="%", row=2, col=2)
fig.show()

summary_payload = {
    "ticker": ticker_symbol,
    "statement_window": {
        "start": str(fundamentals.index[0].date()),
        "end": str(fundamentals.index[-1].date()),
        "years": int(len(fundamentals.index)),
    },
    "latest": {
        "revenue_billions": round(float(latest_fundamentals["revenue_billions"]), 2),
        "ebit_billions": round(float(latest_fundamentals["ebit_billions"]), 2),
        "net_income_billions": round(float(latest_fundamentals["net_income_billions"]), 2),
        "free_cash_flow_billions": round(float(latest_fundamentals["free_cash_flow_billions"]), 2),
        "ebit_margin_pct": round(float(latest_fundamentals["ebit_margin_pct"]), 2),
        "net_margin_pct": round(float(latest_fundamentals["net_margin_pct"]), 2),
        "fcf_margin_pct": round(float(latest_fundamentals["fcf_margin_pct"]), 2),
        "cash_billions": round(float(latest_fundamentals["cash_billions"]), 2),
        "debt_billions": round(float(latest_fundamentals["debt_billions"]), 2),
        "net_debt_billions": round(float(latest_fundamentals["net_debt_billions"]), 2),
    },
    "growth": {
        "revenue_cagr_pct": round(float(cagr_percent(fundamentals["revenue"])), 2),
        "latest_revenue_yoy_pct": round(float(latest_fundamentals["revenue_yoy_pct"]), 2),
        "latest_net_income_yoy_pct": round(float(latest_fundamentals["net_income_yoy_pct"]), 2),
        "latest_free_cash_flow_yoy_pct": round(float(latest_fundamentals["free_cash_flow_yoy_pct"]), 2),
    },
    "health_scorecard": scorecard.assign(value=scorecard["value"].round(2)).to_dict(orient="records"),
    "plain_english": {
        "summary": "Healthy businesses tend to grow revenue, defend margins, convert earnings into cash, and avoid unstable leverage.",
        "current_read": summary_text,
    },
}

Ticker: IBM
Health window: 2016 to 2025
Revenue CAGR: -1.9%
Latest EBIT margin: 18.2%
Latest net margin: 15.7%
Latest free cash flow margin: 17.1%
Cash conversion: 1.25x
Net debt to EBIT: 4.36x
Positive free cash flow years: 10 of 10


,revenue_billions,ebit_billions,net_income_billions,free_cash_flow_billions,revenue_yoy_pct,net_income_yoy_pct,...,ebit_margin_pct,net_margin_pct,fcf_margin_pct,cash_billions,debt_billions,net_debt_billions
fiscalDateEnding,,,,,,,,,,,,,
2016-12-31,79.92,12.96,11.87,12.93,NaN,NaN,...,16.22,14.85,16.18,7.83,42.17,34.34
2017-12-31,79.14,12.02,5.75,12.95,-0.98,-51.54,...,15.18,7.27,16.36,11.97,46.82,34.85
2018-12-31,79.59,12.06,8.73,11.28,0.57,51.71,...,15.16,10.97,14.18,11.38,45.81,34.43
2019-12-31,57.71,8.55,9.43,11.86,-27.49,8.05,...,14.81,16.34,20.55,8.17,64.28,56.11
2020-12-31,73.62,6.01,5.59,15.16,27.56,-40.73,...,8.17,7.59,20.59,13.21,62.90,49.68
2021-12-31,57.35,7.02,5.74,10.42,-22.10,2.72,...,12.24,10.01,18.16,6.65,55.14,48.49
2022-12-31,60.53,2.37,1.64,8.57,5.54,-71.44,...,3.92,2.71,14.17,7.89,54.01,46.13
2023-12-31,61.86,7.51,7.50,11.51,2.20,357.44,...,12.15,12.13,18.61,13.07,59.94,46.87
2024-12-31,62.75,7.51,6.02,11.76,1.44,-19.71,...,11.97,9.60,18.74,13.95,58.40,44.45


,metric,value,unit,status
0,Revenue CAGR,-1.85,%,red
1,Latest EBIT margin,18.16,%,green
2,Cash conversion,1.25,x,green
3,FCF positive years,10.00,/10,green
4,Net debt to EBIT,4.36,x,red


In [17]:
import json
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import yfinance as yf

if fundamentals.empty:
    raise ValueError("Fundamentals data is not available. Run the health dashboard cell first.")

price_company = yf.Ticker(ticker_symbol)
price_history = price_company.history(period="15y", auto_adjust=True)[["Close"]].rename(columns={"Close": "close"})
if price_history.empty:
    raise ValueError(f"No price history was returned for {ticker_symbol}.")

price_history.index = pd.to_datetime(price_history.index).tz_localize(None)
fiscal_dates = pd.to_datetime(fundamentals.index).tz_localize(None)

aligned_prices = price_history["close"].reindex(fiscal_dates, method="ffill")
next_year_prices = price_history["close"].reindex(fiscal_dates + pd.DateOffset(years=1), method="ffill")
prior_year_prices = price_history["close"].reindex(fiscal_dates - pd.DateOffset(years=1), method="ffill")

fiscal_close = aligned_prices.to_numpy(dtype=float)
prior_close = prior_year_prices.to_numpy(dtype=float)
next_close = next_year_prices.to_numpy(dtype=float)

price_model = fundamentals.copy()
price_model["fiscal_close"] = fiscal_close
price_model["trailing_12m_price_return_pct"] = (fiscal_close / prior_close - 1) * 100
price_model["forward_12m_price_return_pct"] = (next_close / fiscal_close - 1) * 100
price_model["ebit_yoy_pct"] = price_model["ebit"].pct_change() * 100
price_model["net_debt_to_ebit"] = price_model["net_debt"] / price_model["ebit"]

analysis_columns = [
    "revenue_yoy_pct",
    "ebit_yoy_pct",
    "net_income_yoy_pct",
    "free_cash_flow_yoy_pct",
    "ebit_margin_pct",
    "fcf_margin_pct",
    "net_debt_to_ebit",
    "trailing_12m_price_return_pct",
    "forward_12m_price_return_pct",
]

analysis_frame = price_model[analysis_columns].replace([np.inf, -np.inf], np.nan).dropna(how="all")
correlation_frame = analysis_frame.dropna()
if len(correlation_frame) < 3:
    raise ValueError("Not enough overlapping annual observations to calculate meaningful price correlations.")

correlation_matrix = correlation_frame.corr().round(2)
stock_correlation = correlation_matrix.loc[
    [
        "revenue_yoy_pct",
        "ebit_yoy_pct",
        "net_income_yoy_pct",
        "free_cash_flow_yoy_pct",
        "ebit_margin_pct",
        "fcf_margin_pct",
        "net_debt_to_ebit",
    ],
    ["trailing_12m_price_return_pct", "forward_12m_price_return_pct"],
]

latest_corr_year = int(correlation_frame.index[-1].year)
summary_lines = [
    f"Ticker: {ticker_symbol}",
    f"Correlation sample: {int(correlation_frame.index[0].year)} to {latest_corr_year}",
    f"Observations used: {len(correlation_frame)} annual points",
]

for return_label in ["trailing_12m_price_return_pct", "forward_12m_price_return_pct"]:
    ranked = stock_correlation[return_label].dropna().sort_values(key=lambda series: series.abs(), ascending=False)
    if not ranked.empty:
        top_metric = ranked.index[0]
        summary_lines.append(
            f"Strongest link to {return_label.replace('_', ' ')}: {top_metric} ({ranked.iloc[0]:.2f})"
        )

print("\n".join(summary_lines))
display(stock_correlation)

display_columns = [
    "revenue_yoy_pct",
    "ebit_yoy_pct",
    "net_income_yoy_pct",
    "free_cash_flow_yoy_pct",
    "ebit_margin_pct",
    "fcf_margin_pct",
    "net_debt_to_ebit",
    "trailing_12m_price_return_pct",
    "forward_12m_price_return_pct",
]
display(price_model[display_columns].round(2))

heatmap = px.imshow(
    stock_correlation,
    text_auto=True,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title=f"{ticker_symbol} Fundamentals vs Stock Return Correlation",
    labels=dict(color="Correlation", x="Stock return window", y="Fundamental metric"),
)
heatmap.update_layout(height=500)
heatmap.show()

scatter_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("EBIT YoY vs Forward 12M Return", "FCF YoY vs Forward 12M Return"),
)

scatter_pairs = [
    ("ebit_yoy_pct", "EBIT YoY"),
    ("free_cash_flow_yoy_pct", "FCF YoY"),
]

for index, (column, label) in enumerate(scatter_pairs, start=1):
    scatter_data = price_model[[column, "forward_12m_price_return_pct"]].dropna()
    scatter_fig.add_trace(
        go.Scatter(
            x=scatter_data[column],
            y=scatter_data["forward_12m_price_return_pct"],
            mode="markers+text",
            text=[str(value.year) for value in scatter_data.index],
            textposition="top center",
            name=label,
            showlegend=False,
        ),
        row=1,
        col=index,
    )
    scatter_fig.update_xaxes(title_text=f"{label} (%)", row=1, col=index)
    scatter_fig.update_yaxes(title_text="Forward 12M price return (%)", row=1, col=index)

scatter_fig.update_layout(height=450, title=f"{ticker_symbol} Fundamental Momentum vs Next-Year Stock Return")
scatter_fig.show()

summary_payload["price_correlation"] = {
    "sample_start": str(correlation_frame.index[0].date()),
    "sample_end": str(correlation_frame.index[-1].date()),
    "observations": int(len(correlation_frame)),
    "matrix": {
        row: {column: round(float(value), 2) for column, value in values.items()}
        for row, values in stock_correlation.to_dict(orient="index").items()
    },
}

print("PRICE_CORRELATION_JSON")
print(json.dumps(summary_payload["price_correlation"], indent=2))

Ticker: IBM
Correlation sample: 2017 to 2025
Observations used: 9 annual points
Strongest link to trailing 12m price return pct: fcf_margin_pct (0.44)
Strongest link to forward 12m price return pct: ebit_margin_pct (-0.50)


,trailing_12m_price_return_pct,forward_12m_price_return_pct
revenue_yoy_pct,-0.20,0.19
ebit_yoy_pct,0.29,0.28
net_income_yoy_pct,0.20,0.38
free_cash_flow_yoy_pct,0.11,0.23
ebit_margin_pct,0.13,-0.50
fcf_margin_pct,0.44,0.03
net_debt_to_ebit,0.08,0.29


,revenue_yoy_pct,ebit_yoy_pct,net_income_yoy_pct,free_cash_flow_yoy_pct,ebit_margin_pct,fcf_margin_pct,net_debt_to_ebit,trailing_12m_price_return_pct,forward_12m_price_return_pct
fiscalDateEnding,,,,,,,,,
2016-12-31,NaN,NaN,NaN,NaN,16.22,16.18,2.65,25.21,-3.99
2017-12-31,-0.98,-7.29,-51.54,0.13,15.18,16.36,2.90,-3.99,-22.56
2018-12-31,0.57,0.42,51.71,-12.88,15.16,14.18,2.85,-22.56,23.58
2019-12-31,-27.49,-29.13,8.05,5.14,14.81,20.55,6.56,23.58,-1.16
2020-12-31,27.56,-29.66,-40.73,27.75,8.17,20.59,8.26,-1.16,16.65
2021-12-31,-22.10,16.74,2.72,-31.28,12.24,18.16,6.91,16.65,10.64
2022-12-31,5.54,-66.22,-71.44,-17.67,3.92,14.17,19.45,10.64,21.85
2023-12-31,2.20,216.78,357.44,34.27,12.15,18.61,6.24,21.85,39.27
2024-12-31,1.44,-0.07,-19.71,2.14,11.97,18.74,5.92,39.27,38.23


PRICE_CORRELATION_JSON
{
  "sample_start": "2017-12-31",
  "sample_end": "2025-12-31",
  "observations": 9,
  "matrix": {
    "revenue_yoy_pct": {
      "trailing_12m_price_return_pct": -0.2,
      "forward_12m_price_return_pct": 0.19
    },
    "ebit_yoy_pct": {
      "trailing_12m_price_return_pct": 0.29,
      "forward_12m_price_return_pct": 0.28
    },
    "net_income_yoy_pct": {
      "trailing_12m_price_return_pct": 0.2,
      "forward_12m_price_return_pct": 0.38
    },
    "free_cash_flow_yoy_pct": {
      "trailing_12m_price_return_pct": 0.11,
      "forward_12m_price_return_pct": 0.23
    },
    "ebit_margin_pct": {
      "trailing_12m_price_return_pct": 0.13,
      "forward_12m_price_return_pct": -0.5
    },
    "fcf_margin_pct": {
      "trailing_12m_price_return_pct": 0.44,
      "forward_12m_price_return_pct": 0.03
    },
    "net_debt_to_ebit": {
      "trailing_12m_price_return_pct": 0.08,
      "forward_12m_price_return_pct": 0.29
    }
  }
}
